# PA005: High Customer Identification (Insiders)

1. Problema de Negócio
    - Selecionar os clientes mais valiosos para integrar um programa de Fidelização



2. Conjunto de Dados
    - Vendas de um e-commerce online, durante o período de 1 ano.

# 0.0. Imports

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import regex as re

import umap.umap_ as umap

from matplotlib import pyplot as plt

from sklearn import cluster as c
from sklearn import metrics as m
from sklearn import preprocessing as pp
from sklearn import decomposition as dd
from sklearn import mixture as mx
from sklearn.manifold import TSNE

from plotly import express as px

## 0.1. Helper Functions

In [2]:
def jupyter_settings():
    %matplotlib inline
    %pylab inline
    
    plt.style.use( 'ggplot' )
    plt.rcParams['figure.figsize'] = [24, 9]
    plt.rcParams['font.size'] = 24
    
    #display( HTML( '<style>.container { width:100% !important; }</style>') )
    pd.options.display.max_columns = None
    pd.options.display.max_rows = None
    pd.set_option( 'display.expand_frame_repr', False )
    
    sns.set()

jupyter_settings()

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


## 0.2. Load dataset

In [3]:
# load data
df_raw = pd.read_csv( '../data/raw/Ecommerce.csv', encoding='ISO-8859-1')

# drop extra column
df_raw.drop( columns=['Unnamed: 8'], axis=1, inplace = True)

In [4]:
df_raw.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,29-Nov-16,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,29-Nov-16,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,29-Nov-16,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,29-Nov-16,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,29-Nov-16,3.39,17850.0,United Kingdom


#  <font color='red'> 1.0. Descrição dos Dados </font>

In [5]:
df1 = df_raw.copy()

In [6]:
df1.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,29-Nov-16,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,29-Nov-16,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,29-Nov-16,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,29-Nov-16,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,29-Nov-16,3.39,17850.0,United Kingdom


## 1.1. Rename Columns

In [7]:
cols_new = ['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country']
df1.columns = cols_new

## 1.2. Data Dimensions

In [8]:
print( 'Number of rows: {}'.format( df1.shape[0] ) )
print( 'Number of cols: {}'.format( df1.shape[1] ) )

Number of rows: 541909
Number of cols: 8


## 1.3. Data Types

In [9]:
df1.dtypes

invoice_no       object
stock_code       object
description      object
quantity          int64
invoice_date     object
unit_price      float64
customer_id     float64
country          object
dtype: object

## 1.4. Check NA

In [10]:
df1.isna().sum()

invoice_no           0
stock_code           0
description       1454
quantity             0
invoice_date         0
unit_price           0
customer_id     135080
country              0
dtype: int64

## 1.5. Replace NA

In [11]:
df_missing = df1.loc[df1['customer_id'].isna(), :]
df_not_missing = df1.loc[~df1['customer_id'].isna(), :]

In [12]:
# create reference
df_backup = pd.DataFrame( df_missing['invoice_no'].drop_duplicates() )
df_backup['customer_id'] = np.arange(19000, 19000 + len(df_backup), 1)

# merge original with reference dataframe
df1 = pd.merge(df1, df_backup, how='left', on='invoice_no')

# coalesce
df1['customer_id'] = df1['customer_id_x'].combine_first(df1['customer_id_y'])

# drop extra columns
df1 = df1.drop(columns=['customer_id_x', 'customer_id_y'], axis=1)

In [13]:
df1.isna().sum()

invoice_no         0
stock_code         0
description     1454
quantity           0
invoice_date       0
unit_price         0
country            0
customer_id        0
dtype: int64

## 1.6. Change dtypes

In [14]:
df1.dtypes

invoice_no       object
stock_code       object
description      object
quantity          int64
invoice_date     object
unit_price      float64
country          object
customer_id     float64
dtype: object

In [15]:
# invoice_date
df1['invoice_date'] = pd.to_datetime( df1['invoice_date'], format='%d-%b-%y' )

# customer_id
df1['customer_id'] = df1['customer_id'].astype(int)

df1.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,country,customer_id
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2016-11-29,2.55,United Kingdom,17850
1,536365,71053,WHITE METAL LANTERN,6,2016-11-29,3.39,United Kingdom,17850
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2016-11-29,2.75,United Kingdom,17850
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2016-11-29,3.39,United Kingdom,17850
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2016-11-29,3.39,United Kingdom,17850


## 1.7. Descriptive Statistics

In [16]:
num_attributes = df1.select_dtypes( include=['int64', 'float64'] )
cat_attributes = df1.select_dtypes( exclude=['int64', 'float64', 'datetime64[ns]'])

### 1.7.1. Numerical Attributes

In [17]:
# Central tendency - mean, median
ct1 = pd.DataFrame( num_attributes.apply( np.mean ) ).T
ct2 = pd.DataFrame( num_attributes.apply( np.median ) ).T

# Dispersion - desvio padrão, min, max, range, std, skew, kurtosis
d1 = pd.DataFrame( num_attributes.apply( np.std ) ).T
d2 = pd.DataFrame( num_attributes.apply( np.min ) ).T
d3 = pd.DataFrame( num_attributes.apply( np.max ) ).T
d4 = pd.DataFrame( num_attributes.apply( lambda x: x.max() - x.min() ) ).T
d5 = pd.DataFrame( num_attributes.apply( lambda x: x.skew() ) ).T
d6 = pd.DataFrame( num_attributes.apply( lambda x: x.kurtosis() ) ).T

# concatenate
m1 = pd.concat( [d2, d3, d4, ct1, ct2, d1, d5, d6] ).T.reset_index()
m1.columns = ['attributes', 'min', 'max', 'range', 'mean', 'mediana', 'std', 'skew', 'kurtosis']
m1

,attributes,min,max,range,mean,mediana,std,skew,kurtosis
0,quantity,-80995.00,80995.0,161990.00,9.552250,3.00,218.080957,-0.264076,119769.160031
1,unit_price,-11062.06,38970.0,50032.06,4.611114,2.08,96.759764,186.506972,59005.719097
2,customer_id,12346.00,22709.0,10363.00,16688.840453,16249.00,2911.408666,0.487449,-0.804287


### 1.7.2. Categorical Attributes

In [18]:
cat_attributes.head()

,invoice_no,stock_code,description,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,United Kingdom
1,536365,71053,WHITE METAL LANTERN,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,United Kingdom


<b>invoice_no</b>

In [19]:
# Problema: Temos invoice com letras e numeros
# df1['invoice_no'].astype( int )

# identificacao
df_letter_invoices = df1.loc[df1['invoice_no'].apply( lambda x: bool( re.search( '[^0-9]+', x) ) ), :]
df_letter_invoices.head()

print( 'Total number of invoices: {}'.format( len( df_letter_invoices )))
print( 'Total number of negative quantity: {}'.format( len( df_letter_invoices[ df_letter_invoices['quantity'] < 0 ] )))

#len( cat_attributes.loc[cat_attributes['invoice_no'].apply( lambda x: bool( re.search( '[^0-9]+', x) ) ), 'invoice_no'].drop_duplicates() )

Total number of invoices: 9291
Total number of negative quantity: 9288


<b>stock_code</b>

In [20]:
#check stock codes only characteres 
df1.loc[cat_attributes['stock_code'].apply( lambda x: bool( re.search( '^[a-zA-Z]+$', x) ) ), 'stock_code'].unique()

# Ação:
## 1. Remove stock_code in ['POST', 'D', 'M', 'PADS', 'DOT', 'CRUK']

array(['POST', 'D', 'DOT', 'M', 'S', 'AMAZONFEE', 'm', 'DCGSSBOY',
       'DCGSSGIRL', 'PADS', 'B', 'CRUK'], dtype=object)

<b>description</b>

In [21]:
df1.head()

# Ação: Delete description

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,country,customer_id
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2016-11-29,2.55,United Kingdom,17850
1,536365,71053,WHITE METAL LANTERN,6,2016-11-29,3.39,United Kingdom,17850
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2016-11-29,2.75,United Kingdom,17850
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2016-11-29,3.39,United Kingdom,17850
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2016-11-29,3.39,United Kingdom,17850


<b>country</b>

In [22]:
len( df1['country'].unique() )

38

In [23]:
df1['country'].value_counts( normalize=True )

United Kingdom          0.914320
Germany                 0.017521
France                  0.015790
EIRE                    0.015124
Spain                   0.004674
Netherlands             0.004375
Belgium                 0.003818
Switzerland             0.003694
Portugal                0.002803
Australia               0.002323
Norway                  0.002004
Italy                   0.001482
Channel Islands         0.001399
Finland                 0.001283
Cyprus                  0.001148
Sweden                  0.000853
Unspecified             0.000823
Austria                 0.000740
Denmark                 0.000718
Japan                   0.000661
Poland                  0.000629
Israel                  0.000548
USA                     0.000537
Hong Kong               0.000531
Singapore               0.000423
Iceland                 0.000336
Canada                  0.000279
Greece                  0.000269
Malta                   0.000234
United Arab Emirates    0.000125
European C

In [24]:
df1[['customer_id', 'country']].drop_duplicates().groupby( 'country' ).count().reset_index().sort_values( 'customer_id', ascending = False)

,country,customer_id
36,United Kingdom,7587
14,Germany,95
13,France,90
10,EIRE,44
31,Spain,31
3,Belgium,25
33,Switzerland,24
27,Portugal,20
19,Italy,15
16,Hong Kong,15


#  <font color='red'> 2.0. Filtragem de Variáveis</font>

In [25]:
df2 = df1.copy()

In [26]:
df2.dtypes

invoice_no              object
stock_code              object
description             object
quantity                 int64
invoice_date    datetime64[ns]
unit_price             float64
country                 object
customer_id              int64
dtype: object

In [27]:
# === Numerical Attributes ======

# unit price > 0.0
df2 = df2.loc[df2['unit_price'] >= 0.04, :]

# === Categorical Attributes ===
df2 = df2[~df2['stock_code'].isin( ['POST', 'D', 'DOT', 'M', 'S', 'AMAZONFEE', 'm', 'DCGSSBOY', 'DCGSSGIRL', 'PADS', 'B', 'CRUK'] )]

# description
df2 = df2.drop( columns='description', axis=1)

# map
df2 = df2[~df2['country'].isin( ['European Community', 'Unspecified'] )]

# bad users
df2 = df2[~df2['customer_id'].isin( [16446])]

## quantity - Negative numbers means products returns
df2_returns = df2.loc[df2['quantity'] < 0, :]
df2_purchases = df2.loc[df2['quantity'] >= 0, :]

#  <font color='red'> 3.0. Feature Engeneering </font>

In [28]:
df3 = df2.copy()

## 3.1. Feature Creation

In [29]:
# data reference
df_ref = df3.drop( ['invoice_no', 'stock_code', 'quantity', 'invoice_date', 'unit_price',
                    'country'], axis=1 ).drop_duplicates( ignore_index = True)

In [30]:
df_ref.head(10)

,customer_id
0,17850
1,13047
2,12583
3,13748
4,15100
5,15291
6,14688
7,17809
8,15311
9,16098


### 3.1.1. Gross Revenue

In [31]:
# Gross Revenue (Faturamento) quanity * price = Faturamento
df2_purchases.loc[:, 'gross_revenue'] = df2_purchases.loc[:, 'quantity'] * df2_purchases.loc[:, 'unit_price']

# Monetary
df_monetary = df2_purchases.loc[:, ['customer_id', 'gross_revenue']].groupby( 'customer_id' ).sum().reset_index()
df_ref = pd.merge( df_ref, df_monetary, on='customer_id', how='left')
df_ref.isna().sum()

/tmp/ipykernel_53974/204081654.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2_purchases.loc[:, 'gross_revenue'] = df2_purchases.loc[:, 'quantity'] * df2_purchases.loc[:, 'unit_price']


customer_id       0
gross_revenue    91
dtype: int64

### 3.1.2. Recency - Day from last purchase

In [32]:
# Recency
df_recency = df2_purchases[['customer_id', 'invoice_date']].groupby( 'customer_id' ).max().reset_index()
df_recency['recency_days'] = ( df3['invoice_date'].max() - df_recency['invoice_date'] ).dt.days
df_recency = df_recency[['customer_id', 'recency_days']].copy()
df_ref = pd.merge( df_ref, df_recency, on='customer_id', how='left' )
df_ref.isna().sum()

customer_id       0
gross_revenue    91
recency_days     91
dtype: int64

### 3.1.4. Quantity of products purchased

In [33]:
# Número de produtos
df_freq = ( df2_purchases[['customer_id', 'stock_code']].groupby( 'customer_id' ).count().reset_index()
                                                        .rename( columns={'stock_code': 'qtde_products' }) )
df_ref = pd.merge( df_ref, df_freq, on='customer_id', how='left' )
df_ref.isna().sum()

customer_id       0
gross_revenue    91
recency_days     91
qtde_products    91
dtype: int64

### 3.1.7. Frequency Purchase

In [34]:
df_aux = ( df2[['customer_id', 'invoice_no', 'invoice_date']].drop_duplicates()
                                                            .groupby( 'customer_id' )
                                                            .agg( max_ = ( 'invoice_date', 'max' ),
                                                                  min_ = ( 'invoice_date', 'min' ),
                                                                  days_ = ( 'invoice_date', lambda x: ((x.max() - x.min()).days ) + 1 ),
                                                                  buy_ = ( 'invoice_no', 'count') ) ).reset_index()                                                               

# Frequency
df_aux['frequency'] = df_aux[['buy_', 'days_']].apply( lambda x: x['buy_'] / x['days_'] if x['days_'] != 0 else 0, axis = 1)

# Merge
df_ref = pd.merge(df_ref, df_aux[['customer_id', 'frequency']], on='customer_id', how='left')

In [35]:
df_aux.sort_values( 'frequency', ascending=False ).head()

,customer_id,max_,min_,days_,buy_,frequency
2777,16120,2017-10-09,2017-10-09,1,5,5.0
2020,15098,2017-06-08,2017-06-08,1,4,4.0
2687,16000,2017-12-05,2017-12-05,1,3,3.0
2635,15920,2017-07-06,2017-07-06,1,3,3.0
3812,17545,2017-10-25,2017-10-25,1,3,3.0


### 3.1.8. Number of Returns

In [36]:
# returns
df_returns = df2_returns[['customer_id', 'quantity']].groupby( 'customer_id' ).sum().reset_index().rename( columns={'quantity':'qtde_returns'})
df_returns['qtde_returns'] = df_returns['qtde_returns'] * -1
df_ref = pd.merge( df_ref, df_returns, on='customer_id', how='left' )

df_ref.loc[df_ref['qtde_returns'].isna(), 'qtde_returns'] = 0

df_ref.isna().sum()


customer_id       0
gross_revenue    91
recency_days     91
qtde_products    91
frequency         0
qtde_returns      0
dtype: int64

#  <font color='red'> 4.0. EDA ( Exploratory Data Analysis )</font>

In [37]:
df4 = df_ref.dropna().copy()

## 4.3. Estudo do Espaço

In [38]:
# original dataset
#df43 = df4.drop( columns=['customer_id'], axis=1 ).copy()

# selected dataset
cols_selected = ['customer_id', 'gross_revenue', 'recency_days', 'qtde_products', 'frequency', 'qtde_returns']
df43 = df4[cols_selected].copy()

In [39]:
import pickle

In [40]:
mm = pp.MinMaxScaler()

df43['gross_revenue'] = mm.fit_transform( df43[['gross_revenue']] )
pickle.dump( mm, open('../src/features/gross_revenue_scaler.pkl', 'wb'), protocol=pickle.HIGHEST_PROTOCOL)

df43['recency_days'] = mm.fit_transform( df43[['recency_days']] )
pickle.dump( mm, open('../src/features/recency_days_scaler.pkl', 'wb'), protocol=pickle.HIGHEST_PROTOCOL)

df43['qtde_products'] = mm.fit_transform( df43[['qtde_products']] )
pickle.dump( mm, open('../src/features/qtde_products_scaler.pkl', 'wb'), protocol=pickle.HIGHEST_PROTOCOL)

df43['frequency'] = mm.fit_transform( df43[['frequency']] )
pickle.dump( mm, open('../src/features/frequency_scaler.pkl', 'wb'), protocol=pickle.HIGHEST_PROTOCOL)

df43['qtde_returns'] = mm.fit_transform( df43[['qtde_returns']] )
pickle.dump( mm, open('../src/features/qtde_returns_scaler.pkl', 'wb'), protocol=pickle.HIGHEST_PROTOCOL)

X = df43.copy()

In [41]:
X.shape

(5695, 6)

In [42]:
X.head()

,customer_id,gross_revenue,recency_days,qtde_products,frequency,qtde_returns
0,17850,0.019312,0.997319,0.037770,0.096237,0.000539
1,13047,0.011579,0.150134,0.021692,0.008676,0.000472
2,12583,0.024020,0.005362,0.029476,0.008059,0.000674
3,13748,0.003396,0.254692,0.003445,0.002497,0.000000
4,15100,0.003137,0.892761,0.000255,0.026211,0.000296


### 4.3.4. Tree-Based Embedding

In [43]:
from sklearn import ensemble as en

In [44]:
# training datasets

X = df43.drop( columns=['customer_id', 'gross_revenue'], axis=1)
y = df43['gross_revenue']

# model definition
rf_model = en.RandomForestRegressor( n_estimators=100, random_state=42 )

# model training
rf_model.fit( X, y)

pickle.dump(rf_model, open('../src/models/rf_model.pkl', 'wb'), protocol=pickle.HIGHEST_PROTOCOL)



In [45]:
df_leaf = pd.DataFrame( rf_model.apply(X) )

In [46]:
df_leaf.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99
0,6432,6884,6930,6402,6583,6823,6851,6914,6957,6502,6557,6446,6639,6802,6606,6403,6523,6699,6849,6875,6700,6782,6811,6942,6823,6857,6563,6680,6524,6644,6615,6672,6665,6544,6893,6982,6709,6948,6421,6885,6824,6521,6928,6600,6728,6529,6887,6845,6934,6978,6643,6834,6496,6437,6716,6913,6453,6635,6740,6425,6580,6835,6809,6584,6506,6920,6478,6599,6853,6941,6887,6875,6385,6497,6827,6687,6835,6531,6953,6900,6769,6812,6931,6557,6558,6824,6542,6855,6692,6537,6923,6515,6991,6719,6704,6649,6496,6541,6574,6834
1,6415,6204,6323,6316,6431,6555,6295,6076,6816,6478,6446,6336,6309,6762,6410,6350,6434,5976,6295,5974,6566,6185,6350,6350,5995,6059,6457,6584,6436,6636,6485,6557,6519,6511,6771,6823,5748,6815,6338,6063,6403,6451,6056,6470,6497,6332,6339,6296,6441,6749,6521,6381,6446,6376,6004,6768,6368,6548,6710,6370,6407,6769,6649,6552,6409,6396,6407,6524,6715,6488,6482,6266,6318,6458,6666,6598,6124,6382,6220,6297,6235,5616,6714,6429,6484,6646,6395,6814,5860,6500,6372,6378,6867,6314,6263,6355,6464,6401,6405,6625
2,5521,6456,6828,6346,6555,6770,6330,6848,6873,6897,6504,6437,6326,6793,6594,6391,6474,6762,6289,6806,6684,6211,6371,6461,6736,6743,6473,6938,6481,5674,6870,6628,6562,5553,6834,6889,6654,6849,6411,6809,5283,6479,6834,6534,6616,6471,6418,6461,6813,6714,6607,5064,6463,5661,6907,6802,6420,6552,6699,5233,6562,6815,6747,5798,6478,6373,6476,6564,6774,6869,6796,6369,6375,4976,6685,6643,6670,6517,6376,6420,6844,6716,6826,6448,6580,6733,6500,6860,6647,6495,6392,6406,6898,5709,6307,6593,6431,6378,6552,6748
3,2773,530,938,664,522,639,1066,437,460,710,1009,3404,1395,960,790,660,574,813,631,494,727,556,926,682,891,3200,415,1101,467,518,384,1096,975,445,335,357,470,1304,1067,287,773,581,1165,851,626,994,865,399,462,862,702,1383,618,257,539,928,1161,501,993,524,1299,680,1059,836,723,997,1297,755,701,954,798,2894,1492,733,996,1020,444,671,1042,740,750,359,554,735,683,3340,696,316,806,1075,1301,827,1100,504,1064,791,695,1142,510,505
4,645,2322,1583,1273,712,2399,1741,2300,2128,2049,2194,567,2125,2351,1816,1191,2363,2160,1373,1754,1133,2050,1742,2779,2601,453,1951,579,1937,828,917,2143,2260,754,1699,1947,2210,2647,2296,706,1107,665,1766,1758,458,522,2351,2273,1747,1341,1665,2171,77,1764,2331,1268,1794,654,1615,2581,2194,1548,2298,1726,1759,2875,2221,506,1125,1661,1807,611,2721,2290,1617,2473,2326,2198,2085,1768,469,2689,795,2211,2302,1108,903,705,2157,1737,2133,1618,635,1718,636,590,1717,2091,1733,2217


In [47]:
reducer = umap.UMAP( random_state=42 )
embedding = reducer.fit_transform( df_leaf )

pickle.dump( reducer, open('../src/features/umap_reducer.pkl', 'wb'), protocol=pickle.HIGHEST_PROTOCOL)


df_tree = pd.DataFrame()
df_tree['embedding_x'] = embedding[:, 0]
df_tree['embedding_y'] = embedding[:, 1]


/home/gabri/miniconda3/envs/pa005/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/gabri/miniconda3/envs/pa005/lib/python3.10/site-packages/sklearn/manifold/_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


#  <font color='red'> 7.0. Hyperparameter Fine-Tunning </font>

In [48]:
X = df_tree.copy()

In [49]:
X.head()

,embedding_x,embedding_y
0,-5.803640,10.739402
1,-7.718688,7.601786
2,-5.611232,10.429735
3,17.176357,-7.422256
4,18.682158,4.169983


#  <font color='red'> 8.0. Model Training </font>

## 8.1. K-Means

In [50]:
k = 8

# model definition
gmm_model = mx.GaussianMixture( n_components=k, random_state=42 )

# model training
gmm_model.fit( X )

# model predict
labels = gmm_model.predict( X ) 

## 8.2. Cluster Validation

In [51]:
## WSS (Within-cluster sum of square)
#print( 'WSS value: {}'.format( kmeans.inertia_ ) )

## SS (Silhouette Score)
print( 'SS value: {}'. format( m.silhouette_score( X, labels, metric='euclidean' ) ) )

SS value: 0.44988784193992615


#  <font color='red'> 9.0. Cluster Analysis </font>

In [52]:
X.head()

,embedding_x,embedding_y
0,-5.803640,10.739402
1,-7.718688,7.601786
2,-5.611232,10.429735
3,17.176357,-7.422256
4,18.682158,4.169983


In [53]:
df9 = X.copy()
df9['cluster'] = labels

In [54]:
#df9 = df4.copy()
#df9['cluster'] = labels
df9.head()

,embedding_x,embedding_y,cluster
0,-5.803640,10.739402,1
1,-7.718688,7.601786,1
2,-5.611232,10.429735,1
3,17.176357,-7.422256,7
4,18.682158,4.169983,0


## 9.2. Cluster Profile

In [55]:
df92 = df4[ cols_selected].copy()
df92['cluster'] = labels

#change dtypes
df92['recency_days'] = df92['recency_days'].astype( int )
df92['qtde_products'] = df92['qtde_products'].astype( int )
df92['qtde_returns'] = df92['qtde_returns'].astype( int )

df92.head()

,customer_id,gross_revenue,recency_days,qtde_products,frequency,qtde_returns,cluster
0,17850,5391.21,372,297,0.486111,40,1
1,13047,3232.59,56,171,0.048780,35,1
2,12583,6705.38,2,232,0.045699,50,1
3,13748,948.25,95,28,0.017921,0,7
4,15100,876.00,333,3,0.136364,22,0


In [56]:
# Number of customer
df_cluster = df92[['customer_id', 'cluster']].groupby( 'cluster' ).count().reset_index()
df_cluster['perc_customer'] = 100 * (df_cluster['customer_id'] / df_cluster['customer_id'].sum())

# Avg Gross revenue
df_avg_gross_revenue = df92[['gross_revenue', 'cluster']].groupby( 'cluster' ).mean().reset_index()
df_cluster = pd.merge( df_cluster, df_avg_gross_revenue, how='inner', on='cluster')

# Avg receny days
df_avg_recency_days = df92[['recency_days', 'cluster']].groupby( 'cluster' ).mean().reset_index()
df_cluster = pd.merge( df_cluster, df_avg_recency_days, how='inner', on='cluster')

# Avg invoice no
df_avg_invoice_no= df92[['qtde_products', 'cluster']].groupby( 'cluster' ).mean().reset_index()
df_cluster = pd.merge( df_cluster, df_avg_invoice_no, how='inner', on='cluster')

# frequency
df_frequency= df92[['frequency', 'cluster']].groupby( 'cluster' ).mean().reset_index()
df_cluster = pd.merge( df_cluster, df_frequency, how='inner', on='cluster')

# returns
df_qtde_returns= df92[['qtde_returns', 'cluster']].groupby( 'cluster' ).mean().reset_index()
df_cluster = pd.merge( df_cluster, df_qtde_returns, how='inner', on='cluster')


df_cluster.sort_values(by='gross_revenue', ascending=False)

,cluster,customer_id,perc_customer,gross_revenue,recency_days,qtde_products,frequency,qtde_returns
1,1,919,16.136962,6230.046213,58.022851,288.441785,0.254723,169.312296
3,3,790,13.871817,2319.979481,97.487342,175.134177,0.407321,3.063291
6,6,640,11.237928,1264.448625,78.009375,57.359375,0.181616,9.784375
7,7,515,9.043020,1031.550369,77.908738,39.504854,0.046137,12.266019
5,5,673,11.817384,834.850282,154.392273,63.331352,0.960927,1.696880
0,0,948,16.646181,336.620949,147.648734,7.795359,0.531971,2.668776
4,4,706,12.396839,295.351629,158.837110,16.001416,0.976193,0.691218
2,2,504,8.849868,231.105913,177.579365,11.136905,0.888462,6.579365


In [57]:
# 7 Cluster Insiders
# 6 Cluster More Products
# 4 Cluster Spend Money
# 5 Cluster Even More Products
# 3 Cluster Less Days
# 2 Cluster Less 1K
# 1 Cluster Stop Returns
# 0 Cluster More Buy

### Cluster 01: (Candidato a Insider)
    - Número de customers: 468 ( 16% do customers )
    - Faturamento médio: 8836
    - Recência média: 21 dias
    - Média de Produtos Comprados: 424 produtos
    - Frequência de produtos comprados: 0.09 produtos/dia 
    - Receita em média: $ 8.836,13 dólares

### Cluster 02:
    - Número de customers: 31 ( 0.71% do customers )
    - Recência média: 14 dias
    - Compras em média: 53 compras
    - Receita média: $40.543,52 dólares

### Cluster 03:
    - Número de customers: 4.335 ( 99% do customers )
    - Recência média: 92 dias
    - Compras em média: 5 compras
    - Receita média: $1.372,57 dólares

#  <font color='red'> 11.0. Deploy to Production </font>

In [58]:
df92.dtypes

customer_id        int64
gross_revenue    float64
recency_days       int64
qtde_products      int64
frequency        float64
qtde_returns       int64
cluster            int64
dtype: object

## 11.1. Insert into SQLITE

In [59]:
import sqlite3
from sqlalchemy import create_engine

In [60]:
# create table
query_create_table_insiders = """
    CREATE TABLE insiders (
        customer_id      INTEGER,
        gross_revenue    REAL,
        recency_days     INTEGER,
        qtde_products    INTEGER,
        frequency        REAL,
        qtde_returns     INTEGER,
        cluster          INTEGER
    )

"""

conn = sqlite3.connect( 'insiders_db.sqlite' )
conn.execute( query_create_table_insiders )
conn.commit()
conn.close()

# insert data
conn = create_engine( 'sqlite:///insiders_db.sqlite')
df92.to_sql( 'insiders', con=conn, if_exists='append', index=False )

OperationalError: table insiders already exists

In [ ]:
# consulting database
query = """
    SELECT * FROM insiders
"""

df = pd.read_sql_query( query, conn )

In [ ]:
df.head()

In [ ]:
df.shape